# 65 — P10.7 SPIDER: preflight, auditoría de gradaciones y congelamiento de alcance

    **Objetivo:** verificar qué gradaciones radiológicas existen realmente, auditar que P10.7 no haya
    sido entrenado/cerrado previamente, congelar las tareas soportadas y crear splits por paciente.

    Este notebook **no entrena**.


> **Gobernanza obligatoria**
>
> - No reentrena estenosis central, foraminal ni subarticular: esas tareas P10.6 ya tienen checkpoints.
> - No accede al test oculto de SPIDER.
> - No usa el `internal_test` para seleccionar modelo o ajustar hiperparámetros.
> - Antes de escribir resultados audita notebooks, manifests, resultados, modelos y todos los `.pt`.
> - Si detecta un export final/frozen previo de P10.7, aborta.
> - La salida es de investigación, requiere revisión profesional y no constituye diagnóstico clínico.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

SEED = 2026
np.random.seed(SEED)

@dataclass(frozen=True)
class Config:
    pfi_root: Path
    spider_root: Path
    images_root: Path
    masks_root: Path
    overview_csv: Path
    gradings_csv: Path
    results_root: Path
    models_root: Path

PFI_ROOT = Path(os.getenv("PFI_ROOT", "/content/drive/MyDrive/PFI_MVP"))
SPIDER_ROOT = Path(os.getenv("PFI_SPIDER_ROOT", str(PFI_ROOT / "data" / "SPIDER")))

def resolve_dir(base: Path, candidates: list[str]) -> Path:
    for rel in candidates:
        path = base / rel
        if path.is_dir():
            return path
    return base / candidates[0]

CFG = Config(
    pfi_root=PFI_ROOT,
    spider_root=SPIDER_ROOT,
    images_root=Path(os.getenv(
        "PFI_SPIDER_IMAGES",
        str(resolve_dir(SPIDER_ROOT, ["images", "images/images"]))
    )),
    masks_root=Path(os.getenv(
        "PFI_SPIDER_MASKS",
        str(resolve_dir(SPIDER_ROOT, ["masks", "masks/masks"]))
    )),
    overview_csv=Path(os.getenv(
        "PFI_SPIDER_OVERVIEW",
        str(SPIDER_ROOT / "overview.csv")
    )),
    gradings_csv=Path(os.getenv(
        "PFI_SPIDER_GRADINGS",
        str(SPIDER_ROOT / "radiological_gradings.csv")
    )),
    results_root=Path(os.getenv(
        "PFI_P10_7_RESULTS_ROOT",
        str(PFI_ROOT / "results" / "P10_7_spider_degenerative")
    )),
    models_root=Path(os.getenv(
        "PFI_P10_7_MODELS_ROOT",
        str(PFI_ROOT / "models" / "P10_7_spider_degenerative")
    )),
)
CFG


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(tmp, path)

REQUIRED_FILES = {
    "overview_csv": CFG.overview_csv,
    "gradings_csv": CFG.gradings_csv,
}
REQUIRED_DIRS = {
    "images_root": CFG.images_root,
    "masks_root": CFG.masks_root,
}

missing = {
    **{name: str(path) for name, path in REQUIRED_FILES.items() if not path.is_file()},
    **{name: str(path) for name, path in REQUIRED_DIRS.items() if not path.is_dir()},
}
if missing:
    raise FileNotFoundError(
        "Faltan insumos SPIDER. No se continúa:\n" +
        json.dumps(missing, indent=2, ensure_ascii=False)
    )

print("Rutas verificadas:")
for name, path in {**REQUIRED_DIRS, **REQUIRED_FILES}.items():
    print(f"- {name}: {path}")


In [ ]:
# Auditoría obligatoria de artefactos antes de crear cualquier salida.
all_pt = sorted(CFG.pfi_root.joinpath("models").rglob("*.pt"))
audit_rows = []
for path in all_pt:
    audit_rows.append({
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "belongs_to_p10_7": CFG.models_root in path.parents,
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

final_tokens = ("final", "frozen", "research_export", "closed")
p10_7_final_pt = [
    path for path in all_pt
    if CFG.models_root in path.parents
    and any(token in path.name.lower() for token in final_tokens)
]
closure_markers = [
    CFG.results_root / "P10_7_CLOSED.json",
    CFG.results_root / "P10_7_RESEARCH_EXPORT_COMPLETE.json",
    CFG.models_root / "P10_7_CLOSED.json",
]
existing_markers = [path for path in closure_markers if path.exists()]

if p10_7_final_pt or existing_markers:
    raise RuntimeError(
        "P10.7 ya parece entrenado/exportado. Se aborta para no sobrescribirlo.\n"
        f"final_pt={list(map(str, p10_7_final_pt))}\n"
        f"markers={list(map(str, existing_markers))}"
    )

print("AUDIT_OK: no hay export final/frozen previo de P10.7.")
print("Los checkpoints P10.6 existentes quedan excluidos del nuevo entrenamiento.")


In [ ]:

overview = pd.read_csv(CFG.overview_csv)
gradings = pd.read_csv(CFG.gradings_csv, dtype={"Patient": str})

print("overview.csv:", overview.shape)
print("radiological_gradings.csv:", gradings.shape)
print("\nColumnas overview:")
print(list(overview.columns))
print("\nColumnas gradings:")
print(list(gradings.columns))

REQUIRED_GRADING_COLUMNS = [
    "Patient",
    "IVD label",
    "Modic",
    "UP endplate",
    "LOW endplate",
    "Spondylolisthesis",
    "Disc herniation",
    "Disc narrowing",
    "Disc bulging",
    "Pfirrman grade",
]
missing_cols = [col for col in REQUIRED_GRADING_COLUMNS if col not in gradings.columns]
if missing_cols:
    raise ValueError(
        "El schema real de radiological_gradings.csv no coincide con el esperado. "
        f"Faltan: {missing_cols}. No se inventan mappings."
    )

print("SCHEMA_GRADINGS_OK")


In [ ]:

def study_id_from_filename(value: Any) -> str:
    name = Path(str(value)).stem
    match = re.match(r"^(\d+)", name)
    if not match:
        raise ValueError(f"No se pudo obtener study_id desde un nombre de archivo del overview.")
    return match.group(1)

overview_name_col = next(
    (c for c in ["new_file_name", "file_name", "filename", "image"] if c in overview.columns),
    None,
)
overview_subset_col = next(
    (c for c in ["subset", "split", "set"] if c in overview.columns),
    None,
)
if overview_name_col is None or overview_subset_col is None:
    raise ValueError(
        "overview.csv debe contener una columna de archivo y una columna subset/split. "
        f"Columnas observadas: {list(overview.columns)}"
    )

overview_map = (
    overview[[overview_name_col, overview_subset_col]]
    .assign(Patient=lambda df: df[overview_name_col].map(study_id_from_filename))
    .assign(
        official_subset=lambda df:
        df[overview_subset_col].astype(str).str.strip().str.lower()
    )
    [["Patient", "official_subset"]]
    .drop_duplicates()
)

if overview_map.groupby("Patient")["official_subset"].nunique().max() > 1:
    raise ValueError("Un mismo paciente aparece en más de un subset oficial.")

patient_table = (
    gradings[["Patient"]]
    .drop_duplicates()
    .merge(overview_map, on="Patient", how="left", validate="one_to_one")
)

if patient_table["official_subset"].isna().any():
    missing_count = int(patient_table["official_subset"].isna().sum())
    raise ValueError(
        f"Hay {missing_count} pacientes de gradings sin subset oficial en overview.csv."
    )

official_train = patient_table[
    patient_table["official_subset"].isin(["training", "train"])
].copy()
official_validation = patient_table[
    patient_table["official_subset"].isin(["validation", "val"])
].copy()

if official_train.empty or official_validation.empty:
    raise ValueError(
        "No se reconocieron subsets train/validation en overview.csv. "
        f"Valores observados: {sorted(patient_table['official_subset'].unique())}"
    )

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
_, dev_val_idx = next(
    splitter.split(official_train, groups=official_train["Patient"])
)

official_train["p10_7_split"] = "dev_train"
official_train.iloc[
    dev_val_idx,
    official_train.columns.get_loc("p10_7_split")
] = "dev_val"

official_validation["p10_7_split"] = "internal_test"

split_df = pd.concat(
    [official_train, official_validation],
    ignore_index=True,
)
split_df = split_df.sort_values(
    ["p10_7_split", "Patient"]
).reset_index(drop=True)

print("Split por pacientes:")
print(split_df["p10_7_split"].value_counts().to_string())
print("INTERNAL_TEST_SEALED = true")


In [ ]:
# Alcance congelado: únicamente gradaciones existentes en SPIDER.
# Las tareas RSNA P10.6 se bloquean para evitar reentrenamiento.
TASKS = {
    "pfirrmann_grade": {
        "source_column": "Pfirrman grade",
        "kind": "ordinal_multiclass",
        "classes": [1, 2, 3, 4, 5],
        "display_name": "Degeneración discal (Pfirrmann)",
    },
    "modic_change": {
        "source_column": "Modic",
        "kind": "multiclass",
        "classes": ["none", "I", "II", "III"],
        "display_name": "Cambios Modic",
    },
    "upper_endplate_change": {
        "source_column": "UP endplate",
        "kind": "binary",
        "classes": [0, 1],
        "display_name": "Cambio de platillo superior / nodo de Schmorl",
    },
    "lower_endplate_change": {
        "source_column": "LOW endplate",
        "kind": "binary",
        "classes": [0, 1],
        "display_name": "Cambio de platillo inferior / nodo de Schmorl",
    },
    "spondylolisthesis": {
        "source_column": "Spondylolisthesis",
        "kind": "binary",
        "classes": [0, 1],
        "display_name": "Espondilolistesis",
    },
    "disc_herniation": {
        "source_column": "Disc herniation",
        "kind": "binary",
        "classes": [0, 1],
        "display_name": "Hernia discal",
    },
    "disc_narrowing": {
        "source_column": "Disc narrowing",
        "kind": "binary",
        "classes": [0, 1],
        "display_name": "Estrechamiento discal",
    },
    "disc_bulging": {
        "source_column": "Disc bulging",
        "kind": "binary",
        "classes": [0, 1],
        "display_name": "Abombamiento discal",
    },
}

BLOCKED_ALREADY_TRAINED = [
    "central_canal_stenosis",
    "neural_foraminal_narrowing",
    "subarticular_stenosis",
]
UNSUPPORTED_WITH_CURRENT_LABELS = [
    "disc_protrusion",
    "disc_extrusion",
    "disc_sequestration",
    "tumor",
    "infection",
    "fracture",
]

scope_preview = {
    "schemaVersion": "pfi.p10-7-spider-task-scope.v1",
    "taskFamily": "disc_level_degenerative_multitask",
    "input": {
        "plane": "sagittal",
        "sequences": ["T1", "T2"],
        "representation": "independent 2.5D disc crops with late/shared-encoder fusion",
    },
    "tasks": TASKS,
    "blockedAlreadyTrained": BLOCKED_ALREADY_TRAINED,
    "unsupportedWithCurrentLabels": UNSUPPORTED_WITH_CURRENT_LABELS,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}
print(json.dumps(scope_preview, indent=2, ensure_ascii=False))


In [ ]:

def canonical_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip()).lower()

gradings_with_split = gradings.merge(
    split_df[["Patient", "p10_7_split"]],
    on="Patient",
    how="left",
    validate="many_to_one",
)

if gradings_with_split["p10_7_split"].isna().any():
    raise ValueError("Hay filas de gradings que no pudieron asociarse a un split.")

development_gradings = gradings_with_split[
    gradings_with_split["p10_7_split"].isin(["dev_train", "dev_val"])
].copy()

internal_test_rows = int(
    gradings_with_split["p10_7_split"].eq("internal_test").sum()
)

print("Resumen de etiquetas SOLO para desarrollo:")
print("development rows:", len(development_gradings))
print("internal_test rows sealed:", internal_test_rows)

for col in REQUIRED_GRADING_COLUMNS[2:]:
    counts = (
        development_gradings[col]
        .map(canonical_text)
        .value_counts(dropna=False)
        .sort_index()
    )
    print(f"\n[{col}]")
    print(counts.to_string())

print("\nMissingness SOLO en desarrollo:")
display(
    pd.DataFrame({
        "column": REQUIRED_GRADING_COLUMNS,
        "missing": [
            int(development_gradings[col].isna().sum())
            for col in REQUIRED_GRADING_COLUMNS
        ],
        "blank": [
            int(development_gradings[col].map(canonical_text).eq("").sum())
            for col in REQUIRED_GRADING_COLUMNS
        ],
    })
)

print("INTERNAL_TEST_LABELS_DISPLAYED = false")
print("INTERNAL_TEST_USED_FOR_SELECTION = false")


In [ ]:
CFG.results_root.mkdir(parents=True, exist_ok=True)
CFG.models_root.mkdir(parents=True, exist_ok=True)

scope_path = CFG.results_root / "task_scope_v1.json"
split_path = CFG.results_root / "patient_split_v1.csv"
manifest_path = CFG.results_root / "preflight_manifest_v1.json"
complete_path = CFG.results_root / "NOTEBOOK_65_COMPLETE.json"

atomic_json(scope_path, scope_preview)
split_df.to_csv(split_path, index=False)

preflight_manifest = {
    "schemaVersion": "pfi.p10-7-preflight-manifest.v1",
    "seed": SEED,
    "paths": {key: str(value) for key, value in asdict(CFG).items()},
    "sourceHashes": {
        "overviewCsvSha256": sha256_file(CFG.overview_csv),
        "gradingsCsvSha256": sha256_file(CFG.gradings_csv),
    },
    "counts": {
        "gradingRows": int(len(gradings)),
        "patients": int(split_df["Patient"].nunique()),
        "devTrainPatients": int(split_df["p10_7_split"].eq("dev_train").sum()),
        "devValPatients": int(split_df["p10_7_split"].eq("dev_val").sum()),
        "internalTestPatientsSealed": int(split_df["p10_7_split"].eq("internal_test").sum()),
    },
    "existingPtAudit": audit_rows,
    "status": "PREFLIGHT_AND_SCOPE_FROZEN",
    "internalTestRowsMappedForSealing": int(split_df["p10_7_split"].eq("internal_test").sum()),
    "internalTestLabelsDisplayed": False,
    "internalTestUsedForSelection": False,
    "officialHiddenTestAccessed": False,
}
atomic_json(manifest_path, preflight_manifest)
atomic_json(complete_path, {
    "status": "NOTEBOOK_65_COMPLETE",
    "scopeSha256": sha256_file(scope_path),
    "splitSha256": sha256_file(split_path),
    "internalTestRowsMappedForSealing": int(split_df["p10_7_split"].eq("internal_test").sum()),
    "internalTestLabelsDisplayed": False,
    "internalTestUsedForSelection": False,
})

print("NOTEBOOK_65_COMPLETE")
print("scope:", scope_path)
print("split:", split_path)
print("manifest:", manifest_path)
